In [19]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
spark.sql(

   """ CREATE TABLE IF NOT EXISTS processed_files (
        file_name STRING,
        load_time TIMESTAMP
    )
    USING DELTA
    """
)

StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 29, Finished, Available, Finished, False)

DataFrame[]

In [20]:
spark.sql(

   """ CREATE TABLE IF NOT EXISTS sales_data (
        order_id INT,
        amount float,
        file_name STRING
    )
    USING DELTA
    """
)

StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 30, Finished, Available, Finished, False)

DataFrame[]

In [21]:
# Read files from lakehouse

from pyspark.sql.functions import input_file_name

df = spark.read.option("header",True).csv("Files/sales/")
df = df.withColumn("file_name",input_file_name())

StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 31, Finished, Available, Finished, False)

In [22]:
df.show(truncate=False)

StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 32, Finished, Available, Finished, False)

+--------+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|order_id|amount|file_name                                                                                                                                                                       |
+--------+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|10      |1000  |abfss://2c41762c-e507-4a0e-965d-878374a7a58d@onelake.dfs.fabric.microsoft.com/ea6e0a15-2968-408c-b008-b17308cb9283/Files/sales/sales_202604.csv?version=1777395395963?flength=44|
|11      |1200  |abfss://2c41762c-e507-4a0e-965d-878374a7a58d@onelake.dfs.fabric.microsoft.com/ea6e0a15-2968-408c-b008-b17308cb9283/Files/sales/sales_202604.csv?version=1777395395963?flength=44|
|12      |1300  |abfss://

In [23]:
## Extract Clean File Name

from pyspark.sql.functions import regexp_extract

df = df.withColumn(
    "file_name",regexp_extract("file_name",r'([^/]+$)',1)
)


StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 33, Finished, Available, Finished, False)

In [24]:
df.show()

StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 34, Finished, Available, Finished, False)

+--------+------+--------------------+
|order_id|amount|           file_name|
+--------+------+--------------------+
|      10|  1000|sales_202604.csv?...|
|      11|  1200|sales_202604.csv?...|
|      12|  1300|sales_202604.csv?...|
|       1|   100|sales_202601.csv?...|
|       2|   200|sales_202601.csv?...|
|       3|   150|sales_202601.csv?...|
|       4|   300|sales_202602.csv?...|
|       5|   250|sales_202602.csv?...|
|       6|   400|sales_202602.csv?...|
|       7|   500|sales_202603.csv?...|
|       8|   600|sales_202603.csv?...|
|       9|   550|sales_202603.csv?...|
+--------+------+--------------------+



In [25]:
## Read Already Processed Files

processed_df = spark.sql("SELECT file_name from processed_files")

StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 36, Finished, Available, Finished, False)

In [26]:
processed_df.show()

StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 37, Finished, Available, Finished, False)

+--------------------+
|           file_name|
+--------------------+
|sales_202602.csv?...|
|sales_202603.csv?...|
|sales_202601.csv?...|
+--------------------+



In [27]:
## Filter new files

new_df = df.join(processed_df, on='file_name',how='left_anti')

StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 39, Finished, Available, Finished, False)

In [28]:
new_df.show()

StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 40, Finished, Available, Finished, False)

+--------------------+--------+------+
|           file_name|order_id|amount|
+--------------------+--------+------+
|sales_202604.csv?...|      10|  1000|
|sales_202604.csv?...|      11|  1200|
|sales_202604.csv?...|      12|  1300|
+--------------------+--------+------+



In [29]:
from pyspark.sql.functions import col

new_df = new_df.withColumn("order_id",col("order_id").cast("int")).withColumn("amount",col("amount").cast("float"))

StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 42, Finished, Available, Finished, False)

In [30]:
new_df.show()

StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 43, Finished, Available, Finished, False)

+--------------------+--------+------+
|           file_name|order_id|amount|
+--------------------+--------+------+
|sales_202604.csv?...|      10|1000.0|
|sales_202604.csv?...|      11|1200.0|
|sales_202604.csv?...|      12|1300.0|
+--------------------+--------+------+



In [31]:
new_df.write.format('delta').mode('append').saveAsTable("sales_data")

StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 45, Finished, Available, Finished, False)

In [32]:
sales_data = spark.sql("SELECT * FROM sales_data")

StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 46, Finished, Available, Finished, False)

In [33]:
sales_data.show()

StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 47, Finished, Available, Finished, False)

+--------+------+--------------------+
|order_id|amount|           file_name|
+--------+------+--------------------+
|      10|1000.0|sales_202604.csv?...|
|      11|1200.0|sales_202604.csv?...|
|      12|1300.0|sales_202604.csv?...|
|       4| 300.0|sales_202602.csv?...|
|       5| 250.0|sales_202602.csv?...|
|       6| 400.0|sales_202602.csv?...|
|       7| 500.0|sales_202603.csv?...|
|       8| 600.0|sales_202603.csv?...|
|       9| 550.0|sales_202603.csv?...|
|       1| 100.0|sales_202601.csv?...|
|       2| 200.0|sales_202601.csv?...|
|       3| 150.0|sales_202601.csv?...|
+--------+------+--------------------+



In [34]:
from pyspark.sql.functions import current_timestamp

existing_files = spark.table("sales_data")\
.select("file_name")\
.distinct()\
.withColumn("load_time",current_timestamp())


existing_files.write.format("delta")\
.mode("overwrite")\
.saveAsTable("processed_files")

StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 49, Finished, Available, Finished, False)

In [35]:
spark.sql("Select * from processed_files").show()

StatementMeta(, 25560896-33a9-4fdd-8f2d-a842ee1351c3, 50, Finished, Available, Finished, False)

+--------------------+--------------------+
|           file_name|           load_time|
+--------------------+--------------------+
|sales_202604.csv?...|2026-04-28 16:58:...|
|sales_202602.csv?...|2026-04-28 16:58:...|
|sales_202603.csv?...|2026-04-28 16:58:...|
|sales_202601.csv?...|2026-04-28 16:58:...|
+--------------------+--------------------+

